In [ ]:
import subprocess
import json
from pathlib import Path
import shutil

# ------------- Safety: ensure ffmpeg & ffprobe exist -------------
def check_ffmpeg_installed():
    for exe in ["ffmpeg", "ffprobe"]:
        if shutil.which(exe) is None:
            raise RuntimeError(f"{exe} not found in PATH. Install ffmpeg first.")
    print("ffmpeg & ffprobe found")


# ------------- Run shell commands cleanly -------------
def run_cmd(cmd):
    """Run a shell command and raise a helpful error if it fails."""
    result = subprocess.run(
        cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        text=True
    )
    if result.returncode != 0:
        raise RuntimeError(
            f"Command failed:\n{' '.join(cmd)}\n\nSTDERR:\n{result.stderr}"
        )
    return result.stdout


# ------------- Get video resolution using ffprobe -------------
def get_video_resolution(input_path):
    """
    Returns (width, height) of the first video stream using ffprobe.
    """
    cmd = [
        "ffprobe",
        "-v", "error",
        "-select_streams", "v:0",
        "-show_entries", "stream=width,height",
        "-of", "json",
        str(input_path)
    ]
    out = run_cmd(cmd)
    data = json.loads(out)
    stream = data["streams"][0]
    width = stream["width"]
    height = stream["height"]
    return width, height


# ------------- Decide which resolutions we want -------------
def get_target_heights(input_height):
    """
    Given input video height, decide which heights to generate.
    Always keep original height + downscaled versions only.
    Rules:
      - h <= 320  -> original + 144 (if smaller)
      - 320 < h <= 720 -> original + 480, 320, 144 (only if < h)
      - h > 720   -> original + 720, 480, 320, 144 (only if < h)
    """
    h = input_height
    targets = set()

    # Always keep original
    targets.add(h)

    if h <= 320:
        # "320p case" -> make 144p if possible
        if h > 144:
            targets.add(144)
    elif h <= 720:
        # "720p case" -> 480, 320, 144
        for t in [480, 320, 144]:
            if t < h:   # ensure no upscaling
                targets.add(t)
    else:
        # >720 -> 720, 480, 320, 144 (downscaled only)
        for t in [720, 480, 320, 144]:
            if t < h:
                targets.add(t)

    # Return sorted from highest to lowest (just for nice order)
    return sorted(targets, reverse=True)


# ------------- Generate HLS chunks for one resolution -------------
def generate_hls_for_height(input_path, output_dir, target_height, input_height):
    """
    Generate HLS chunks for a specific resolution.
    If target_height == input_height, we do NOT upscale; we simply
    remux/re-encode to HLS at same resolution (no scale filter).
    Each resolution gets its own folder with:
      - playlist.m3u8
      - chunk_000.ts, chunk_001.ts, ...
    """
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    playlist_path = output_dir / "playlist.m3u8"
    segment_pattern = str(output_dir / "chunk_%03d.ts")

    cmd = [
        "ffmpeg",
        "-y",  # overwrite
        "-i", str(input_path),
    ]

    # Only scale when target_height < input_height (downscale)
    if target_height < input_height:
        # Maintain aspect ratio: scale height = target_height, width = auto
        scale_filter = f"scale=-2:{target_height}"
        cmd += ["-vf", scale_filter]

    # Video & audio encoding settings (you can tune later)
    cmd += [
        "-c:v", "libx264",
        "-preset", "veryfast",
        "-crf", "23",
        "-c:a", "aac",
        "-b:a", "128k",
        "-f", "hls",
        "-hls_time", "4",            # 4-second chunks
        "-hls_playlist_type", "vod",
        "-hls_segment_filename", segment_pattern,
        str(playlist_path)
    ]

    print(f"Generating HLS for {target_height}p in {output_dir} ...")
    run_cmd(cmd)
    print(f"Done: {target_height}p → {playlist_path}")
    return playlist_path


# ------------- Main pipeline: E2E + write .txt -------------
def process_video_local(
    input_video_path,
    base_output_dir="outputs_local",
    index_filename="outputs_index.txt"
):
    """
    Main pipeline:
    - Detect input resolution
    - Decide target resolutions
    - For each height, create HLS chunks in its own folder
    - Save list of output dirs into a .txt file
    """
    check_ffmpeg_installed()

    input_path = Path(input_video_path)
    if not input_path.exists():
        raise FileNotFoundError(f"Input video not found: {input_path}")

    base_output_dir = Path(base_output_dir)
    base_output_dir.mkdir(parents=True, exist_ok=True)

    # 1. Get input resolution
    width, height = get_video_resolution(input_path)
    print(f"Input resolution: {width}x{height} (height={height})")

    # 2. Decide which heights to generate
    target_heights = get_target_heights(height)
    print(f"Target heights: {target_heights}")

    output_dirs = []

    # 3. Generate HLS per resolution
    for h in target_heights:
        folder_name = f"{h}p"
        out_dir = base_output_dir / folder_name
        generate_hls_for_height(
            input_path=input_path,
            output_dir=out_dir,
            target_height=h,
            input_height=height
        )
        output_dirs.append(str(out_dir.resolve()))

    # 4. Save directory paths into .txt
    index_path = base_output_dir / index_filename
    with open(index_path, "w") as f:
        for d in output_dirs:
            f.write(d + "\n")

    print(f"Saved output directories in: {index_path.resolve()}")
    return {
        "input_resolution": (width, height),
        "target_heights": target_heights,
        "output_dirs": output_dirs,
        "index_file": str(index_path.resolve())
    }


In [7]:
result = process_video_local(
    input_video_path="/home/ananta/personnel_study_material/practice-and-projects/Sample_data/video_converter/video_720p.mp4",
    base_output_dir="/home/ananta/personnel_study_material/practice-and-projects/Sample_data/video_converter/",
    index_filename="testing.txt"
)

result

✅ ffmpeg & ffprobe found
📏 Input resolution: 1280x720 (height=720)
🎯 Target heights: [720, 480, 320, 144]
🎬 Generating HLS for 720p in /home/ananta/personnel_study_material/practice-and-projects/Sample_data/video_converter/720p ...
✅ Done: 720p → /home/ananta/personnel_study_material/practice-and-projects/Sample_data/video_converter/720p/playlist.m3u8
🎬 Generating HLS for 480p in /home/ananta/personnel_study_material/practice-and-projects/Sample_data/video_converter/480p ...
✅ Done: 480p → /home/ananta/personnel_study_material/practice-and-projects/Sample_data/video_converter/480p/playlist.m3u8
🎬 Generating HLS for 320p in /home/ananta/personnel_study_material/practice-and-projects/Sample_data/video_converter/320p ...
✅ Done: 320p → /home/ananta/personnel_study_material/practice-and-projects/Sample_data/video_converter/320p/playlist.m3u8
🎬 Generating HLS for 144p in /home/ananta/personnel_study_material/practice-and-projects/Sample_data/video_converter/144p ...
✅ Done: 144p → /home/ana

{'input_resolution': (1280, 720),
 'target_heights': [720, 480, 320, 144],
 'output_dirs': ['/home/ananta/personnel_study_material/practice-and-projects/Sample_data/video_converter/720p',
  '/home/ananta/personnel_study_material/practice-and-projects/Sample_data/video_converter/480p',
  '/home/ananta/personnel_study_material/practice-and-projects/Sample_data/video_converter/320p',
  '/home/ananta/personnel_study_material/practice-and-projects/Sample_data/video_converter/144p'],
 'index_file': '/home/ananta/personnel_study_material/practice-and-projects/Sample_data/video_converter/testing.txt'}

In [ ]:
##  CHANGES REQUIRED TO MAKE AN WORKING API

## Input we will take from get 
## We will use boto3 and an simple api to store over s3
## We will change the index_filename to take mongodb and store the data as per the requirements